# 10 - Repeated small replay corruption under fixed-class traffic refresh (RF, post hoc)
Pre-registration: `experiments/prereg_refresh.md`, committed before any RF code or run. Code: `s14_refresh.py`,
which reuses `s12_repeated.py`'s summarize and test code with its own layout.

**What this tests: traffic refresh with a FIXED class set, not new-class arrival.**
- **E0** = BENIGN + Syn + TFTP.
- **Updates E1-E6** each bring fresh, separate BENIGN and TFTP rows. No class is new, and no fresh Syn rows ever
  arrive, so Syn lives only in the replay buffer, where the attacker works.
- **Why post hoc.** RC (notebook 08) and RCS (notebook 09) both stopped at their clean stage, because class-arrival
  sequences were not stable without poison. RF was designed after seeing that, and its results say nothing about
  new-class arrival.

**Attack (as RC).** Before every update, rho x 2,000 more stored Syn labels are flipped to BENIGN (rho = 10%, then
5%), and the flips persist.

**Arms (as RC):**
- clean;
- poisoned (gate only);
- poisoned_repair (soft R1);
- a forced soft-R1 diagnostic on the poisoned arm.

| Stage | What it runs | Rule |
|---|---|---|
| Strict clean pilot (cell 3) | One schedule, clean arm only, seeds 42-44, validation only | A seed is stable only if all 6 updates are accepted, every update's smallest gate margin is >= 0.01, and Syn val recall stays within 0.02 of A0. **PROCEED only if 3/3 seeds are stable.** Otherwise STOP: no poisoned run, no other schedule |
| Poisoned runs (cell 4) | rho 0.10 (resumes the pilot runs; their clean arm is the control), then rho 0.05 | Fixed 6-update horizon; rho 0.05 is skipped if the T4 time is over 90 min |

Test is scored once (cells 7-9), and only after the validation files are committed in git.

**Inputs (Add Input).** Notebook 04's output, as for notebooks 05-09: `nids_out/` with `clean_*.parquet` and
`runs/cii/all12_cii_s42`. Settings: **Internet ON**, accelerator **GPU T4**.

**Expected time:**
- about 30-40 min of T4;
- about 6 min if the pilot stops.

Run cells one at a time, never with Save & Run All.

**If the session dies:** rerun cell 1 through the last unfinished cell. Every stage resumes: finished runs are
skipped, and a decided pilot prints its saved decision.

In [ ]:
# 1. Locate code and the notebook-04 outputs (read only); everything is written under /kaggle/working/nids_out
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s14_refresh.py")] +
     sorted(glob.glob("/kaggle/input/**/s14_refresh.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "code with s14_refresh.py not found: turn Internet ON"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git)

SRC = X.find_dir("runs/cii/all12_cii_s42/config.json", "NIDS_RC_SRC")
assert SRC and (SRC / "clean_train.parquet").exists(), \
    "attach notebook 04's output: nids_out/ with clean_*.parquet and runs/cii/all12_cii_s42"
WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
RF = WORK / "rf"
print("source (read only):", SRC, "\nwrites:", WORK)

def run(*args):
    cmd = [sys.executable, str(Path(CODE_DIR) / "s14_refresh.py"), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

def show_stage(per_seed):
    rows = [{"seed": s, "t": u["t"], "accepted": u["accepted"], "min_margin": round(u["min_margin"], 3),
             "binding": u["binding"], "Syn": round(u["Syn_recall"], 4)}
            for s, v in per_seed.items() for u in v["updates"]]
    display(pd.DataFrame(rows).pivot_table(index="seed", columns="t", values=["accepted", "min_margin", "Syn"],
                                           aggfunc="first"))
    for s, v in per_seed.items():
        print(f"  seed {s}: stable={v['stable']} | all accepted={v['all_accepted']} | "
              f"min margin={v['min_margin']:.3f} | max |Syn - A0|={v['max_abs_Syn_change']:.4f}")

COMMON = ["--work", WORK]

In [ ]:
# 2. GPU check; copy the Stage 1 files (hash-checked against all12) and build Stage 1b of tag ref6
#    (about 1 min). Refuses if the protected val rows differ from the recorded feasibility check.
#    In a RESTORE session run this AFTER cell 7.
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)
run("prepare", "--src", SRC, *COMMON)

In [ ]:
# 3. STRICT CLEAN PILOT: clean arm only, seeds 42-44, validation only (about 5 min). Always writes rf_clean.zip
#    (the pilot's validation files; no models, no test), on success and on failure.
t0 = time.time()
run("pilot", *COMMON)
p = json.loads((RF / "pilot.json").read_text())
show_stage(p["per_seed"])
print(f"\nPROCEED = {p['proceed']} ({p['reason']}) | {time.time() - t0:.0f} s")
run("zip-val", *COMMON, "--bundle", WORK.parent / "rf_clean.zip")
if not p["proceed"]:
    print("STOP HERE: the clean refresh sequence is not stable. Send rf_clean.zip; do not run cells 4-9 "
          "(no poisoned run, no other schedule)")

In [ ]:
# 4. Poisoned runs, validation only: rho 0.10 (resumes the pilot runs), then rho 0.05 unless the measured T4
#    time > 90 min (about 25-35 min). Refuses unless the pilot said PROCEED. Re-running this cell resumes.
t0 = time.time()
run("run", *COMMON)
print(f"run: {time.time() - t0:.0f} s")

In [ ]:
# 5. Trajectories and the registered predictions, from saved files (validation only)
run("summarize", *COMMON)
traj = pd.read_csv(RF / "trajectory_val.csv")
print("deployed model's Syn val recall after update t:")
display(traj.pivot_table(index=["rho", "seed", "arm"], columns="t", values="Syn_recall").round(3))
print("C1 decision per update (True = accepted):")
display(traj[traj.t > 0].pivot_table(index=["rho", "seed", "arm"], columns="t", values="c1_accepted", aggfunc="first"))

## STOP - download BOTH zips from the Output panel (cell 6 writes them)
- `rf_val.zip` holds the validation JSON/CSV of the pilot and the poisoned runs only: no buffers, models or test.
  Send it; it is committed under `experiments/results/10_rf/`.
- `rf_restore.zip` holds the deployed models A0..A6, the validation files and a sha256 manifest, with no test.
  Keep it for a new session.

Test is scored once, and only after that commit: set `DECISIONS_COMMIT` in cell 8, then run cells 8-9.

If this session has closed, start a new one and run these cells, never cells 3-6:
1. cell 1;
2. cell 7, with `RESTORE = True` and the restore zip attached;
3. cell 2;
4. cells 8 and 9.

In [ ]:
# 6. Validation outputs: rf_val.zip (JSON/CSV only) and rf_restore.zip (deployed models + manifest); no test
assert not (RF / "test").exists(), "test was already scored in this folder"
run("zip-val", *COMMON, "--bundle", WORK.parent / "rf_val.zip")
run("bundle", *COMMON, "--bundle", WORK.parent / "rf_restore.zip")

In [ ]:
# 7. RESTORE (only in a NEW session after the old one closed; skip otherwise). Then run cell 2, then cells 8-9.
RESTORE = False
if RESTORE:
    man = [m for m in sorted(glob.glob("/kaggle/input/**/RESTORE_MANIFEST.json", recursive=True))
           if (Path(m).parent / "rf").is_dir()] or sorted(glob.glob("/kaggle/input/**/rf_restore.zip", recursive=True))
    assert man, "attach the dataset made from rf_restore.zip"
    b = Path(man[0])
    run("restore", *COMMON, "--bundle", b.parent if b.name == "RESTORE_MANIFEST.json" else b)
else:
    print("restore skipped (RESTORE = False)")

In [ ]:
# 8. Test, ONCE. Disabled until DECISIONS_COMMIT is the sha of the commit that holds rf_val.zip's files.
#    score-test refuses unless every validation file here is byte-identical to that commit's copy under
#    experiments/results/10_rf/ and every deployed model has the hash recorded in rf/summary.json.
DECISIONS_COMMIT = ""
if DECISIONS_COMMIT:
    subprocess.run(["git", "-C", CODE_DIR, "fetch", "--depth", "50", "origin", "main"], check=True)
    run("score-test", *COMMON, "--decisions-commit", DECISIONS_COMMIT, "--repo", CODE_DIR)
else:
    print("skipped: commit the validation files first, then set DECISIONS_COMMIT")

In [ ]:
# 9. Test files (after cell 8 ran; original or restore session)
tdir = RF / "test"
if not tdir.exists():
    print("no test scored yet")
else:
    dest = WORK.parent / "rf_test.zip"
    pick = sorted(p for p in tdir.rglob("*") if p.is_file())
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in pick:
            zf.write(p, p.relative_to(WORK))
    print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.2f} MB). Download it from the Output panel.")